In [ ]:
import csv
import json
import time
from typing import List, Dict
from confluent_kafka import Producer
from confluent_kafka.error import KafkaError
import pyarrow.parquet as pq
import pyarrow as pa


In [ ]:
conf = {
    'bootstrap.servers': 'localhost:9092',
    'client.id': 'yellow_taxi_record_api', 

    # Reliability (defaults usually fine)
    'acks': 'all',              # Wait for all replicas to acknowledge then gives ok status(-1)
    'enable.idempotence': True,  # Exactly-once delivery, if duplicate message comes ignore the same 

    # Performance
    'linger.ms': 5,             # Batch messages (default 0) , this means it waits 5ms before requesting 
    'batch.size': 16384,        # 16KB batches (default)
    'compression.type': 'snappy', # Compress the message to almost 75% 
}
producer = Producer(conf)

In [ ]:
green_taxi = pq.ParquetFile('/Users/dhananjayojha/Documents/projects/de_projects/nyc_taxi_cc/data/green_taxi/green_tripdata_2025-01.parquet')

In [ ]:
# logic to ingest data to kafka broker 

index = 1
for batch in green_taxi.iter_batches(batch_size=1000): 
    chunk_df = pa.Table.from_batches([batch]).to_pandas()
    records = chunk_df.to_dict('records')
    print(f"Sending Batch : {index}")
    index += 1 
    # Send to Kafka
    print("Sending following records :")
    print(records[:3])
    for record in records:
        producer.produce('yellow_taxi_bookings', value=json.dumps(record, default=str).encode())
    producer.flush()
    print(f"Sent {len(records)} records")